In [ ]:
merged_output_dir = "/Volumes/T7/enriched_merged"
energy_excluded_dir = "/Volumes/T7/enriched_excluding_energy"
os.makedirs(energy_excluded_dir, exist_ok=True)

import glob, os
parquet_files = [
    f for f in glob.glob(os.path.join(merged_output_dir, "*.parquet"))
    if not os.path.basename(f).startswith("._")
]
enriched_ddf = dd.read_parquet(parquet_files)

print("Loaded merged dataset from disk.")

# 2) Filter out BIDTYPE=ENERGY
filtered_ddf = enriched_ddf[enriched_ddf["BIDTYPE"] != "ENERGY"]
print("Filtering out BIDTYPE == 'ENERGY'...")

# 3) Write to a new folder
print(f"Writing non-ENERGY data to {energy_excluded_dir} ...")
filtered_ddf.to_parquet(energy_excluded_dir, overwrite=True)
print("Non-ENERGY filter complete!")

In [ ]:
import dask.dataframe as dd
import os

# Define paths
sample_file = '/Volumes/T7/enriched_excluding_energy/part.2.parquet'
output_dir = '/Volumes/T7/enriched_sa1_sample/filtered_sample.parquet/part.0.parquet'
output_file = os.path.join(output_dir, 'filtered_sample.parquet')

# Step 1: Read the sample file
ddf = dd.read_parquet(sample_file)
print("Sample file loaded.")

# Step 2: Inspect the data
print("Data types:\n", ddf.dtypes)
print("First few rows:\n", ddf.head())

# Step 3: Apply filtering
filtered_ddf = ddf[(ddf['BIDTYPE'] != 'ENERGY') & (ddf['Region'] == 'SA1')]
print("Filtering applied.")

# Step 4: Verify the filtered data
print("Filtered data types:\n", filtered_ddf.dtypes)
print("First few rows of filtered data:\n", filtered_ddf.head())

# Step 5: Write the filtered data
filtered_ddf.to_parquet(output_file, overwrite=True)
print(f"Filtered data written to {output_file}")

# Step 6: Review the output
review_ddf = dd.read_parquet(output_file)
print("Filtered file reloaded.")
print("First few rows of the filtered file:\n", review_ddf.head())

In [ ]:
import glob
import os
import time
import dask.dataframe as dd
import pandas as pd
from datetime import datetime

def load_valid_parquet_files(folder_pattern):
    """
    Load all valid parquet files matching the given pattern into a Dask DataFrame.
    Excludes hidden files (like macOS ._* files).
    
    Args:
        folder_pattern: Glob pattern to match parquet files
        
    Returns:
        Dask DataFrame containing the data from all valid parquet files
    """
    # Get all files matching the pattern
    all_files = glob.glob(folder_pattern)
    
    # Filter out hidden files (like macOS ._ files)
    valid_files = [f for f in all_files if not os.path.basename(f).startswith('._')]
    
    # Sort files to ensure consistent reading order
    valid_files = sorted(valid_files)
    
    print(f"Found {len(valid_files)} valid parquet files from pattern: {folder_pattern}")
    
    if not valid_files:
        raise ValueError(f"No valid parquet files found for pattern: {folder_pattern}")
    
    # Load files into a Dask DataFrame
    return dd.read_parquet(valid_files, engine='pyarrow')

def filter_and_merge_data(volume_ddf, price_ddf, output_dir):
    """
    Filter data for 2017-2018 and merge volume and price data.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print("Converting date columns to datetime format...")
    start_time = time.time()
    
    # Convert SETTLEMENTDATE to datetime
    volume_ddf = volume_ddf.assign(
        SETTLEMENTDATE=dd.to_datetime(volume_ddf["SETTLEMENTDATE"], errors="coerce")
    )
    price_ddf = price_ddf.assign(
        SETTLEMENTDATE=dd.to_datetime(price_ddf["SETTLEMENTDATE"], errors="coerce")
    )
    
    print(f"Date conversion defined in {time.time() - start_time:.2f} seconds")
    
    # Filter for 2017-2018 data
    print("Filtering for 2017-2018 data...")
    start_time = time.time()
    
    start_date = pd.Timestamp('2017-01-01')
    end_date = pd.Timestamp('2018-12-31 23:59:59')
    
    volume_filtered = volume_ddf[
        (volume_ddf['SETTLEMENTDATE'] >= start_date) & 
        (volume_ddf['SETTLEMENTDATE'] <= end_date)
    ]
    
    price_filtered = price_ddf[
        (price_ddf['SETTLEMENTDATE'] >= start_date) & 
        (price_ddf['SETTLEMENTDATE'] <= end_date)
    ]
    
    print(f"Filtering defined in {time.time() - start_time:.2f} seconds")
    
    # Persist the filtered DataFrames to avoid recomputation
    print("Persisting filtered data (this may take some time)...")
    start_time = time.time()
    
    volume_filtered = volume_filtered.persist()
    price_filtered = price_filtered.persist()
    
    # Wait for persist to complete
    vol_npartitions = volume_filtered.npartitions
    price_npartitions = price_filtered.npartitions
    
    print(f"Filtered volume data: {vol_npartitions} partitions")
    print(f"Filtered price data: {price_npartitions} partitions")
    print(f"Persistence completed in {time.time() - start_time:.2f} seconds")
    
    # Process one partition at a time
    for i in range(vol_npartitions):
        print(f"Processing volume partition {i+1}/{vol_npartitions}...")
        start_time = time.time()
        
        try:
            # Get one volume partition as pandas DataFrame
            vol_part = volume_filtered.get_partition(i).compute()
            print(f"Loaded volume partition with {len(vol_part)} rows in {time.time() - start_time:.2f} seconds")
            
            # Process each price partition
            for j in range(price_npartitions):
                print(f"  Processing price partition {j+1}/{price_npartitions}...")
                part_start = time.time()
                
                try:
                    # Get one price partition as pandas DataFrame
                    price_part = price_filtered.get_partition(j).compute()
                    print(f"  Loaded price partition with {len(price_part)} rows in {time.time() - part_start:.2f} seconds")
                    
                    # Merge the DataFrames
                    merge_start = time.time()
                    merged_df = vol_part.merge(
                        price_part,
                        on=["SETTLEMENTDATE", "DUID", "BIDTYPE", "BIDBAND"],
                        how="inner",
                        suffixes=('_volume', '_price')
                    )
                    print(f"  Merged to {len(merged_df)} rows in {time.time() - merge_start:.2f} seconds")
                    
                    # If we have merged data, write it to a parquet file
                    if not merged_df.empty:
                        save_start = time.time()
                        output_file = os.path.join(output_dir, f"merged_2017_2018_v{i}_p{j}.parquet")
                        merged_df.to_parquet(output_file, engine='pyarrow', index=False)
                        print(f"  Wrote {len(merged_df)} rows to {output_file} in {time.time() - save_start:.2f} seconds")
                    else:
                        print(f"  No matching data between these partitions")
                    
                except Exception as e:
                    print(f"  Error processing price partition {j}: {str(e)}")
                    continue
                
        except Exception as e:
            print(f"Error processing volume partition {i}: {str(e)}")
            continue

def main():
    # Set up paths for input files
    volume_pattern = "/Volumes/T7/bid-volume-melted-files-A4/*.parquet"
    price_pattern = "/Volumes/T7/bid-price-melted-files-B3/*.parquet"
    
    print("Loading volume data metadata...")
    volume_ddf = load_valid_parquet_files(volume_pattern)
    print(f"Volume data columns: {list(volume_ddf.columns)}")
    print(f"Volume data partitions: {volume_ddf.npartitions}")
    
    print("Loading price data metadata...")
    price_ddf = load_valid_parquet_files(price_pattern)
    print(f"Price data columns: {list(price_ddf.columns)}")
    print(f"Price data partitions: {price_ddf.npartitions}")
    
    # Create output directory if it doesn't exist
    output_dir = "/Volumes/T7/bid-merged-fcas-2017-2018"
    print(f"Will save merged data to {output_dir}...")
    
    # Filter and merge data
    filter_and_merge_data(volume_ddf, price_ddf, output_dir)
    
    # Report completion
    print("All done!")
    print(f"Output saved to: {output_dir}")

if __name__ == "__main__":
    main()

In [ ]:
import glob
import os
import dask.dataframe as dd

merged_output_dir = "/Volumes/T7/enriched_excluding_energy"
sa1_output_dir = "/Volumes/T7/enriched_sa1"
os.makedirs(sa1_output_dir, exist_ok=True)

# Collect all Parquet files, excluding hidden ones
parquet_files = [
    f for f in glob.glob(os.path.join(merged_output_dir, "*.parquet"))
    if not os.path.basename(f).startswith("._")
]

# Ensure there are valid Parquet files to process
if not parquet_files:
    print("No valid Parquet files found in the specified directory.")
else:
    # Read the Parquet files into a Dask DataFrame
    enriched_ddf = dd.read_parquet(parquet_files)
    print("Loaded merged dataset from disk.")

    # Filter for Region=SA1
    sa_ddf = enriched_ddf[enriched_ddf["Region"] == "SA1"]
    print("Filtering for Region == 'SA1'...")

    # Write the filtered data to a new folder
    print(f"Writing SA1 data to {sa1_output_dir} ...")
    sa_ddf.to_parquet(sa1_output_dir, overwrite=True)
    print("SA1 filter complete!")

In [ ]:
# # Filter for rows where INTERVAL_DATETIME contains the bids for the 6-6:05pm time interval

# def filter_interval_time(input_dir, output_dir, target_time='18:05:00'):
#     """
#     Filter parquet files for rows where INTERVAL_DATETIME contains a specific time (e.g., '18:05:00')
    
#     Parameters:
#     -----------
#     input_dir : str
#         Directory containing the parquet files to filter
#     output_dir : str
#         Directory where filtered files will be saved
#     target_time : str
#         Time to filter for in format 'HH:MM:SS'
#     """
#     print(f"Starting to filter data for interval time: {target_time}")
    
#     # Create the output directory if it doesn't exist
#     os.makedirs(output_dir, exist_ok=True)
#     print(f"Output directory created/verified: {output_dir}")
    
#     # Get list of all parquet files in the input directory
#     file_list = glob.glob(os.path.join(input_dir, "*.parquet"))
    
#     # Sort the file list to ensure consistent processing order
#     file_list.sort()
    
#     print(f"Found {len(file_list)} parquet files to process")
    
#     # Counter for tracking progress
#     total_files = len(file_list)
#     files_processed = 0
#     rows_filtered = 0
#     total_rows_processed = 0
#     files_with_target = 0
    
#     for file_path in file_list:
#         file_name = os.path.basename(file_path)
#         files_processed += 1
        
#         print(f"Processing file {files_processed}/{total_files}: {file_name}")
        
#         try:
#             # Read the parquet file
#             df = pd.read_parquet(file_path)
            
#             # Track total rows for statistics
#             file_row_count = len(df)
#             total_rows_processed += file_row_count
            
#             # Filter for rows where INTERVAL_DATETIME contains the target time
#             if 'INTERVAL_DATETIME' in df.columns:
#                 # Check data type and filter accordingly
#                 if pd.api.types.is_datetime64_any_dtype(df['INTERVAL_DATETIME']):
#                     # If it's a datetime column, extract components
#                     hour, minute, second = map(int, target_time.split(':'))
#                     mask = (df['INTERVAL_DATETIME'].dt.hour == hour) & \
#                            (df['INTERVAL_DATETIME'].dt.minute == minute) & \
#                            (df['INTERVAL_DATETIME'].dt.second == second)
#                 else:
#                     # If it's a string column, use string contains method
#                     mask = df['INTERVAL_DATETIME'].astype(str).str.contains(target_time)
                
#                 filtered_df = df[mask]
                
#                 # Count filtered rows for reporting
#                 filtered_row_count = len(filtered_df)
#                 rows_filtered += filtered_row_count
                
#                 # Only write output if we have rows that match
#                 if filtered_row_count > 0:
#                     files_with_target += 1
                    
#                     # Construct output file path
#                     output_file_path = os.path.join(output_dir, file_name)
                    
#                     # Save the filtered DataFrame to a new parquet file
#                     filtered_df.to_parquet(output_file_path, index=False)
#                     print(f"  - Saved {filtered_row_count} rows with {target_time} to {output_file_path}")
#                 else:
#                     print(f"  - No rows with {target_time} found in this file")
#             else:
#                 print(f"  - Warning: INTERVAL_DATETIME column not found in {file_name}")
                
#         except Exception as e:
#             print(f"  - Error processing {file_name}: {str(e)}")
    
#     # Print summary statistics
#     print("\nProcessing complete!")
#     print(f"Processed {total_files} files with {total_rows_processed} total rows")
#     print(f"Found {rows_filtered} rows containing interval time {target_time}")
#     print(f"Found target time in {files_with_target} out of {total_files} files")
#     print(f"Filtered data saved to {output_dir}")

# if __name__ == "__main__":
#     # Directories
#     input_directory = "/Volumes/T7/bid-volume-filtered-2"
#     output_directory = "/Volumes/T7/bid-volume-filtered-3"
    
#     # Run the filter function for the 18:05:00 time period
#     filter_interval_time(
#         input_dir=input_directory,
#         output_dir=output_directory,
#         target_time='18:05:00'
#     )

In [ ]:
# # Filter feather files for RaiseReg and LowerReg BIDTYPEs and save as parquet files
# # Create the output directory if it doesn't already exist
# output_dir = '/Volumes/T7/bid-volume-filtered-2'
# os.makedirs(output_dir, exist_ok=True)

# file_list = glob.glob('/Volumes/T7/bid-volume-data-sorted/*.feather')

# # Sort alphabetically by filename
# file_list.sort()

# for file in file_list:
#     print(f"Processing {file}...")
#     # Read the Feather file
#     df = pd.read_feather(file)
    
#     # Filter to keep rows where BIDTYPE is either "RAISEREG" or "LOWERREG"
#     filtered_volume_bids = df[(df["BIDTYPE"] == "RAISEREG") | (df["BIDTYPE"] == "LOWERREG")]

#     # Print the first few rows of the filtered DataFrame
#     print("Filtered DataFrame head:")
#     print(filtered_volume_bids.head())

#     # Get just the filename without the path
#     base_filename = os.path.basename(file)  # e.g. "PUBLIC_DVD_BIDPEROFFER_D_201201010000.feather"
#     # Remove '.feather' extension
#     filename_no_ext = os.path.splitext(base_filename)[0]  # e.g. "PUBLIC_DVD_BIDPEROFFER_D_201201010000"

#     # Construct the full output path in output_dir
#     out_file = os.path.join(output_dir, filename_no_ext + ".parquet")

#     # Save to Parquet
#     filtered_volume_bids.to_parquet(out_file, index=False)
#     print(f"Finished processing {file} -> {out_file}\n")